# Modelos predictores no lineales: árbol y bosque

En este notebook copmparamos el desempeño predictivo del árbol de regresión y el bosque aleatorio. Usaremos los datosde salario sy empleo para Barranquilla. En ejercicios pasados observamos que MCO sobreajusta cuando la dimensionalidad crece, y mostramos cómo Lasso mejora la predicción en estas circunstancias. 

In [ ]:
suppressWarnings(
pacman::p_load(xtable, dplyr, caret, rpart, RColorBrewer, rpart.plot, randomForest)
)

In [ ]:
file<-"https://raw.githubusercontent.com/andvarga-eco/econometrics_ml/refs/heads/main/data/geih_24_baq.csv"
data<-read.csv(file)

In [ ]:
data<-data|>filter(P6040 %in% 25:64)|>filter(P3042!=99)
data<-na.omit(data)
data<-data|>filter(P6800<=60)|>mutate(w=INGLABO/(P6800*4),lw=log(w),
                                      edad2=P6040^2, edad3=P6040^3, edad4=P6040^4)
data<-data|>mutate(P3271=factor(P3271), P3042=factor(P3042),
RAMA2D_R4=forcats::fct_lump_n(factor(RAMA2D_R4),n=40,other_level="Otras"))

In [ ]:
set.seed(1234)
training <- sample(nrow(data), nrow(data) * (3 / 4), replace = FALSE)
data_train <- data[training, ]
data_test <- data[-training, ]

## Árbol de regresión

In [ ]:
basic<-lw~(P3271+P6040+P3042+RAMA2D_R4)
tree.fit.basic<-rpart(basic,data=data_train,minbucket=5,cp=0.01)
printcp(tree.fit.basic)
rsq.rpart((tree.fit.basic))

In [ ]:
bestcp <- tree.fit.basic$cptable[which.min(tree.fit.basic$cptable[, "xerror"]), "CP"]
bestcp

In [ ]:
fit_prunedtree <- prune(tree.fit.basic, cp = bestcp)
rsq.rpart(fit_prunedtree)
rpart.plot(fit_prunedtree)

In [ ]:
y_test<-data_test$lw
yhat_pt <- predict(fit_prunedtree, newdata = data_test)
mse_pt <- summary(lm((y_test - yhat_pt)^2 ~ 1))$coef[1:2]
r2_pt <- 1 - mse_pt[1] / var(y_test)
cat("R2:",round(r2_pt,2),
" | MSE:",round(mse_pt[1],2), "\n")

## Bagging

In [ ]:
p <- length(all.vars(basic)) - 1

In [ ]:

fit_bagg <- randomForest(basic, ntree = 2000, nodesize = 20,mtry=p, data = data_train)
fit_bagg
yhat_bagg<-predict(fit_bagg,newdata = data_test)
mse_bagg <- summary(lm((y_test - yhat_bagg)^2 ~ 1))$coef[1:2]
r2_bagg<- 1 - mse_bagg[1] / var(y_test)
cat("R2:",round(r2_bagg,2),
" | MSE:",round(mse_bagg[1],2), "\n")

## Ranfom forest sin Bagging

In [ ]:
fit_rf <- randomForest(basic,ntree = 2000, nodesize = 20, data = data_train)
fit_rf
yhat_rf<-predict(fit_rf,newdata = data_test)
mse_rf<- summary(lm((y_test - yhat_rf)^2 ~ 1))$coef[1:2]
r2_rf<- 1 - mse_rf[1] / var(y_test)
cat("R2:",round(r2_rf,2),
" | MSE:",round(mse_rf[1],2), "\n")